# Junção da base de preços com informações cadastrais e setoriais

Nesta etapa, será feita a junção dos ativos com os preços com a bse de dados contendo os setores, extraidas do Economatica. 

Além disso, será criado um identificador do papel chamado "id_papel", utilizando o ISIN sempre que disponível e o ticker como alternativa quando o ISIN estiver ausente. Essa escolha reduz o risco de tratar uma mudança de ticker como se fosse um ativo diferente.

### Importação das bibliotecas

Nesta etapa, importamos as bibliotecas necessárias para manipular dados, trabalhar com datas, ler arquivos em Parquet e organizar os caminhos dos arquivos do projeto.

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

### Definição dos caminhos dos arquivos

Nesta etapa, definimos os caminhos dos arquivos que serão usados.

A base de entrada é a base de preços já tratada no notebook anterior.

Também definimos o caminho onde será salva a base final de liquidez histórica.

In [36]:
arquivo_precos = Path("../dados_tratados/dados_economatica_B3_tratado.parquet")

arquivo_saida = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

print("Arquivo de preços existe?", arquivo_precos.exists())

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

Arquivo de preços existe? True


### Carregamento da base tratada de preços

Nesta etapa, carregamos a base tratada criada no notebook anterior.

Essa base contém os preços ajustados por proventos, volume financeiro, quantidade de negócios e demais informações necessárias para selecionar os ativos líquidos.

In [37]:
df_tratado = pd.read_parquet(arquivo_precos)

print("Tamanho da base tratada:", df_tratado.shape)

display(df_tratado.head())

Tamanho da base tratada: (1374399, 9)


ativo  fechamento_ajustado  abertura_ajustada  \
ticker data                                                              
AALR3  2016-10-27  AALR3<XBSP>              19.7240            19.7240   
       2016-10-28  AALR3<XBSP>              18.9350            19.0337   
       2016-10-31  AALR3<XBSP>              17.8108            18.9252   
       2016-11-01  AALR3<XBSP>              17.6530            17.8108   
       2016-11-03  AALR3<XBSP>              17.7417            17.7516   

                   minimo_ajustado  maximo_ajustado  medio_ajustado  \
ticker data                                                           
AALR3  2016-10-27          19.7240          19.7240         19.7240   
       2016-10-28          18.6589          19.4873         19.0238   
       2016-10-31          17.2684          18.9350         17.9192   
       2016-11-01          16.9232          18.1263         17.4952   
       2016-11-03          17.0711          17.9883         17.6826   

                      q_negs  volume_financeiro      q_titulos  
ticker data                                                     
AALR3  2016-10-27     0.0000             0.0000         0.0000  
       2016-10-28 4,460.0000   122,334,647.0000 6,342,600.0000  
       2016-10-31 4,238.0000    45,857,231.0000 2,523,300.0000  
       2016-11-01 2,072.0000    17,676,981.0000   996,200.0000  
       2016-11-03 2,157.0000    11,132,994.0000   621,000.0000

### Preparação da base de preços para junção

A base tratada de preços foi salva com índice composto por ticker e data.

Para juntar essa base com as informações cadastrais e setoriais do Economatica, transformamos o índice novamente em colunas.

Essa versão será usada apenas para incorporar informações como ISIN, situação CVM, setor, subsetor e segmento.

In [38]:
df_precos = df_tratado.reset_index()

df_precos["data"] = pd.to_datetime(df_precos["data"], errors="coerce")

df_precos = df_precos.sort_values(["data", "ticker"]).reset_index(drop=True)

display(df_precos.head())

,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos
0,ABCB4,2010-01-04,ABCB4<XBSP>,3.8282,3.7197,3.7197,3.8499,3.8189,407.0000,"2,479,611.0000","201,200.0000"
1,ABEV3,2010-01-04,ABEV3<XBSP>,3.1813,3.0875,3.0565,3.1855,3.1401,94.0000,"4,938,379.0000","817,500.0000"
2,ABYA3,2010-01-04,ABYA3<XBSP>,4.6000,4.6100,4.5700,4.6200,4.6000,463.0000,"3,991,454.0000","868,600.0000"
3,ACGU3,2010-01-04,ACGU3<XBSP>,5.7800,5.6500,5.6000,5.7800,5.7300,875.0000,"4,033,320.0000","704,100.0000"
4,AEDU11,2010-01-04,AEDU11<XBSP>,25.0965,24.8666,24.8666,25.4465,25.2465,"1,359.0000","11,432,550.0000","452,800.0000"


### Validação das colunas necessárias

Antes de calcular liquidez, verificamos se a base possui as colunas necessárias.

As colunas principais são:

- ticker;
- data;
- fechamento ajustado;
- volume financeiro;
- quantidade de negócios.

Sem essas variáveis, não é possível construir corretamente o universo líquido.

In [39]:
colunas_necessarias = [
    "ticker",
    "data",
    "fechamento_ajustado",
    "volume_financeiro",
    "q_negs"
]

colunas_faltantes = [
    col for col in colunas_necessarias
    if col not in df_precos.columns
]

if colunas_faltantes:
    raise ValueError(f"Colunas faltantes na base de preços: {colunas_faltantes}")

print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


### Carregamento da base de setores

Como a formação dos pares será restrita a ativos do mesmo setor, precisamos carregar a informação de setor econômico extraída do Economatica.

Essa base será usada para associar cada ticker ao seu respectivo setor.

A informação de setor não será usada para filtrar liquidez nesta etapa, mas será preservada para a etapa posterior de formação dos pares. 

In [40]:
arquivo_setores = Path("../dados/economatica_B3_setores.csv")

print("Arquivo de setores existe?", arquivo_setores.exists())

Arquivo de setores existe? True


In [41]:
setores = pd.read_csv(
    arquivo_setores,
    sep=",",
    encoding="latin1"
)

print("Tamanho da base de setores:", setores.shape)

display(setores.head())

Tamanho da base de setores: (1419, 10)


,Ativo,Nome,Classe,Código,ISIN,CNPJ,Situação|CVM,Setor Econômico|Bovespa,Subsetor Bovespa,Segmento Bovespa
0,TTEN3<XBSP>,3tentos,ON,TTEN3,BRTTENACNOR0,94813102000170,ATIVO,Consumo não cíclico,Agropecuária,Agricultura
1,QVUM3B<XBSP>,521 Particip,ON,QVUM3B,-,01547749000116,ATIVO,-,-,-
2,QVQP3<XBSP>,524 Particip,ON,QVQP3,BRQVQPACNOR1,01851771000155,ATIVO,Outros,Outros,Outros
3,APPA3<XBSP>,A P Participacoes,ON,APPA3,-,02288752000125,CANCELADA,-,-,-
4,APPA4<XBSP>,A P Participacoes,PN,APPA4,-,02288752000125,CANCELADA,-,-,-


### Padronização das colunas da base de setores

As colunas extraídas do Economatica podem vir com espaços, acentos ou caracteres especiais.

Nesta etapa, padronizamos os nomes das colunas para facilitar o tratamento e evitar erros.

In [42]:
setores.columns = (
    setores.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("|", "_", regex=False)
)

setores.columns

Index(['ativo', 'nome', 'classe', 'código', 'isin', 'cnpj', 'situação_cvm',
       'setor_econômico_bovespa', 'subsetor_bovespa', 'segmento_bovespa'],
      dtype='str')

### Criação do ticker limpo na base de setores

Para juntar corretamente a base de setores com a base de preços, criamos uma coluna de ticker limpo, removendo qualquer informação entre <...>

In [43]:
setores["ativo"] = setores["ativo"].astype(str).str.strip().str.upper()

setores["ticker"] = (
    setores["ativo"]
    .str.replace(r"<.*?>", "", regex=True)
    .str.strip()
)

display(setores[["ativo", "ticker"]].head())

,ativo,ticker
0,TTEN3<XBSP>,TTEN3
1,QVUM3B<XBSP>,QVUM3B
2,QVQP3<XBSP>,QVQP3
3,APPA3<XBSP>,APPA3
4,APPA4<XBSP>,APPA4


### Padronização da coluna de setor

Nessa etapa, fizemos a padronização dos nomes das colunas da base de setores.

O objetivo é remover acentos, espaços e caracteres especiais.

In [44]:
import unicodedata

def remover_acentos(texto):
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    return texto

setores.columns = [
    remover_acentos(col) for col in setores.columns
]

setores.columns = (
    setores.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("|", "_", regex=False)
)

setores.columns

Index(['ativo', 'nome', 'classe', 'codigo', 'isin', 'cnpj', 'situacao_cvm',
       'setor_economico_bovespa', 'subsetor_bovespa', 'segmento_bovespa',
       'ticker'],
      dtype='str')

### Renomeação das colunas setoriais

Nesta etapa, renomeamos as colunas da base cadastral para nomes mais simples.

As principais colunas serão:

- ticker;
- isin;
- situação CVM;
- setor;
- subsetor;
- segmento.

A coluna "ticker" será usada para juntar a base de setores com a base de preços. A coluna "isin" será usada para criar um identificador mais estável do papel.

In [45]:
setores = setores.rename(columns={
    "codigo": "codigo_economatica",
    "situacao_cvm": "situacao_cvm",
    "setor_economico_bovespa": "setor",
    "subsetor_bovespa": "subsetor",
    "segmento_bovespa": "segmento"
})

setores.columns

Index(['ativo', 'nome', 'classe', 'codigo_economatica', 'isin', 'cnpj',
       'situacao_cvm', 'setor', 'subsetor', 'segmento', 'ticker'],
      dtype='str')

### Limpar coluna ISIN

Neste etapa, trocamos os "-" da coluna ISIN por NaN, pois o pandas não interpreta o "-", usado pelo economaica, como valor nulo.

In [46]:
colunas_para_limpar = [
    "isin",
    "cnpj",
    "situacao_cvm",
    "setor",
    "subsetor",
    "segmento"
]

for coluna in colunas_para_limpar:
    if coluna in setores.columns:
        setores[coluna] = (
            setores[coluna]
            .astype(str)
            .str.strip()
            .replace("-", np.nan)
            .replace("", np.nan)
            .replace("nan", np.nan)
            .replace("NaN", np.nan)
            .replace("NAN", np.nan)
            .replace("None", np.nan)
        )

### Criação do identificador do papel

Nesta etapa, criamos a coluna "id_papel"

Sempre que o ISIN estiver disponível, ele será usado como identificador principal do papel. Quando o ISIN estiver ausente, usaremos o ticker como alternativa.

Essa decisão reduz o risco de tratar uma mudança de ticker como se fosse um ativo diferente.

In [47]:
setores["id_papel"] = np.where(
    setores["isin"].notna(),
    setores["isin"],
    setores["ticker"]
)

display(setores[["ticker", "isin", "id_papel", "setor", "subsetor", "segmento"]].head())

,ticker,isin,id_papel,setor,subsetor,segmento
0,TTEN3,BRTTENACNOR0,BRTTENACNOR0,Consumo não cíclico,Agropecuária,Agricultura
1,QVUM3B,NaN,QVUM3B,NaN,NaN,NaN
2,QVQP3,BRQVQPACNOR1,BRQVQPACNOR1,Outros,Outros,Outros
3,APPA3,NaN,APPA3,NaN,NaN,NaN
4,APPA4,NaN,APPA4,NaN,NaN,NaN


### Verificação se restou "-" do economatica

Essa etapa é para conferir se restou alguma célula nula com "-" usual do economatica. É necessário ser 0.

In [48]:
display(
    setores[["ticker", "isin", "id_papel", "setor", "subsetor", "segmento"]].head()
)

print("ISIN nulos:", setores["isin"].isna().sum())
print("Setores nulos:", setores["setor"].isna().sum())
print("id_papel iguais a '-':", (setores["id_papel"] == "-").sum())

,ticker,isin,id_papel,setor,subsetor,segmento
0,TTEN3,BRTTENACNOR0,BRTTENACNOR0,Consumo não cíclico,Agropecuária,Agricultura
1,QVUM3B,NaN,QVUM3B,NaN,NaN,NaN
2,QVQP3,BRQVQPACNOR1,BRQVQPACNOR1,Outros,Outros,Outros
3,APPA3,NaN,APPA3,NaN,NaN,NaN
4,APPA4,NaN,APPA4,NaN,NaN,NaN


ISIN nulos: 507
Setores nulos: 593
id_papel iguais a '-': 0


### Criação da tabela cadastral por ticker

Nesta etapa, criamos uma tabela cadastral com uma única linha por ticker.

Essa tabela será usada para juntar as base de dados dos setores com a base de dados dos preços.

In [49]:
colunas_setores_ticker = [
    "ticker",
    "nome",
    "classe",
    "codigo_economatica",
    "isin",
    "id_papel",
    "cnpj",
    "situacao_cvm",
    "setor",
    "subsetor",
    "segmento"
]

setores_ticker = (
    setores[colunas_setores_ticker]
    .dropna(subset=["ticker"])
    .drop_duplicates(subset=["ticker"], keep="last")
    .copy()
)

print("Quantidade de tickers na base cadastral:", setores_ticker["ticker"].nunique())
print("Quantidade de identificadores de papel:", setores_ticker["id_papel"].nunique())

display(setores_ticker.head())

print("Tickers totais:", setores_ticker["ticker"].nunique())
print("Tickers sem ISIN:", setores_ticker["isin"].isna().sum())
print("Tickers sem setor:", setores_ticker["setor"].isna().sum())
print("Tickers sem subsetor:", setores_ticker["subsetor"].isna().sum())
print("Tickers sem segmento:", setores_ticker["segmento"].isna().sum())

display(setores_ticker["setor"].value_counts(dropna=False).head(20))

Quantidade de tickers na base cadastral: 1419
Quantidade de identificadores de papel: 1419


,ticker,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,TTEN3,3tentos,ON,TTEN3,BRTTENACNOR0,BRTTENACNOR0,94813102000170,ATIVO,Consumo não cíclico,Agropecuária,Agricultura
1,QVUM3B,521 Particip,ON,QVUM3B,NaN,QVUM3B,01547749000116,ATIVO,NaN,NaN,NaN
2,QVQP3,524 Particip,ON,QVQP3,BRQVQPACNOR1,BRQVQPACNOR1,01851771000155,ATIVO,Outros,Outros,Outros
3,APPA3,A P Participacoes,ON,APPA3,NaN,APPA3,02288752000125,CANCELADA,NaN,NaN,NaN
4,APPA4,A P Participacoes,PN,APPA4,NaN,APPA4,02288752000125,CANCELADA,NaN,NaN,NaN


Tickers totais: 1419
Tickers sem ISIN: 507
Tickers sem setor: 593
Tickers sem subsetor: 593
Tickers sem segmento: 593


setor
NaN                               593
Consumo cíclico                   181
Financeiro                        161
Bens industriais                  125
Utilidade pública                 119
Materiais básicos                  73
Consumo não cíclico                44
Saúde                              41
Outros                             26
Tecnologia da informação           22
Petróleo gás e biocombustíveis     18
Comunicações                       16
Name: count, dtype: int64

### Limpeza da coluna ISIN

Após o merge, é necessário limpar a coluna ISIN novamente para trocar os "-" por NaN.

In [50]:
df_precos_setores["isin"] = (
    df_precos_setores["isin"]
    .astype(str)
    .str.strip()
    .replace("-", np.nan)
    .replace("", np.nan)
    .replace("nan", np.nan)
    .replace("NaN", np.nan)
    .replace("NAN", np.nan)
    .replace("None", np.nan)
)

### Verificação se restou "-"

In [51]:
print("id_papel nulos:", df_precos_setores["id_papel"].isna().sum())
print("id_papel iguais a '-':", (df_precos_setores["id_papel"] == "-").sum())

display(
    df_precos_setores[
        df_precos_setores["id_papel"].isna() |
        (df_precos_setores["id_papel"] == "-")
    ][["ticker", "isin", "id_papel", "data", "fechamento_ajustado"]].head(20)
)

id_papel nulos: 0
id_papel iguais a '-': 0


,ticker,isin,id_papel,data,fechamento_ajustado


### Junção da base de setores com a base de preços

Nesta etapa, juntamos as informações cadastrais e setoriais com a base de preços tratada.

A junção será feita pelo ticker, pois a base de preços possui o código de negociação de cada ativo.

In [52]:
linhas_antes = len(df_precos)
tickers_antes = df_precos["ticker"].nunique()

df_precos_setores = df_precos.merge(
    setores_ticker,
    on="ticker",
    how="left"
)

linhas_depois = len(df_precos_setores)
tickers_depois = df_precos_setores["ticker"].nunique()

print("Linhas antes:", linhas_antes)
print("Linhas depois:", linhas_depois)
print("Tickers antes:", tickers_antes)
print("Tickers depois:", tickers_depois)
print("Tickers sem setor:", df_precos_setores[df_precos_setores["setor"].isna()]["ticker"].nunique())
print("Tickers sem ISIN:", df_precos_setores[df_precos_setores["isin"].isna()]["ticker"].nunique())

display(df_precos_setores.head())

Linhas antes: 1374399
Linhas depois: 1374399
Tickers antes: 775
Tickers depois: 775
Tickers sem setor: 122
Tickers sem ISIN: 30


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ABCB4,2010-01-04,ABCB4<XBSP>,3.8282,3.7197,3.7197,3.8499,3.8189,407.0000,"2,479,611.0000","201,200.0000",Abc Brasil,PN,ABCB4,BRABCBACNPR4,BRABCBACNPR4,28195667000106,ATIVO,Financeiro,Intermediários financeiros,Bancos
1,ABEV3,2010-01-04,ABEV3<XBSP>,3.1813,3.0875,3.0565,3.1855,3.1401,94.0000,"4,938,379.0000","817,500.0000",Ambev S/A,ON,ABEV3,BRABEVACNOR1,BRABEVACNOR1,07526557000100,ATIVO,Consumo não cíclico,Bebidas,Cervejas e refrigerantes
2,ABYA3,2010-01-04,ABYA3<XBSP>,4.6000,4.6100,4.5700,4.6200,4.6000,463.0000,"3,991,454.0000","868,600.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
3,ACGU3,2010-01-04,ACGU3<XBSP>,5.7800,5.6500,5.6000,5.7800,5.7300,875.0000,"4,033,320.0000","704,100.0000",Guarani,ON,ACGU3,NaN,ACGU3,47080619000117,CANCELADA,NaN,NaN,NaN
4,AEDU11,2010-01-04,AEDU11<XBSP>,25.0965,24.8666,24.8666,25.4465,25.2465,"1,359.0000","11,432,550.0000","452,800.0000",Anhanguera,UNT N2,AEDU11,NaN,AEDU11,04310392000146,CANCELADA,NaN,NaN,NaN


In [53]:
if linhas_depois != linhas_antes:
    raise ValueError("A quantidade de linhas mudou após o merge. Verifique duplicatas em setores_ticker.")

if tickers_depois != tickers_antes:
    raise ValueError("A quantidade de tickers mudou após o merge. Verifique a junção.")

print("Junção validada: linhas e tickers foram preservados.")

Junção validada: linhas e tickers foram preservados.


### Criação do identificador final do papel na base de preços + setores

Após a junção, garantimos que a base final, de setores e preços, tenha a coluna "id_papel".

Se o ISIN estiver disponível, usamos o ISIN. Caso contrário, usamos o ticker.

In [54]:
df_precos_setores["id_papel"] = np.where(
    df_precos_setores["isin"].notna(),
    df_precos_setores["isin"],
    df_precos_setores["ticker"]
)

display(df_precos_setores[["ticker", "isin", "id_papel", "setor", "subsetor", "segmento"]].head())

,ticker,isin,id_papel,setor,subsetor,segmento
0,ABCB4,BRABCBACNPR4,BRABCBACNPR4,Financeiro,Intermediários financeiros,Bancos
1,ABEV3,BRABEVACNOR1,BRABEVACNOR1,Consumo não cíclico,Bebidas,Cervejas e refrigerantes
2,ABYA3,NaN,ABYA3,NaN,NaN,NaN
3,ACGU3,NaN,ACGU3,NaN,NaN,NaN
4,AEDU11,NaN,AEDU11,NaN,NaN,NaN


### Checagem de múltiplos tickers por papel

Nesta etapa, verificamos se um mesmo "id_papel" aparece associado a mais de um ticker.

Isso pode acontecer em casos de mudança de ticker ao longo dos anos.

In [55]:
tickers_por_papel = (
    df_precos_setores
    .dropna(subset=["id_papel"])
    .groupby("id_papel")["ticker"]
    .nunique()
    .reset_index(name="qtd_tickers")
    .sort_values("qtd_tickers", ascending=False)
)

display(tickers_por_papel.head(20))

,id_papel,qtd_tickers
0,ABYA3,1
1,ACGU3,1
2,AEDU11,1
3,AGEI3,1
4,AGIN3,1
5,ALLL11,1
6,ALLL4,1
7,AVIL3,1
8,BNCA3,1
9,BRAALRACNOR6,1


### Tratamento de duplicatas por papel e data

Como o identificador principal agora é o "id_papel", é preciso garantir que exista apenas uma observação por papel em cada data.

Se houver mais de uma observação para o mesmo "id_papel" na mesma data, mantemos a linha com maior volume financeiro.

In [56]:
duplicatas_id_data = df_precos_setores.duplicated(
    subset=["id_papel", "data"]
).sum()

print("Duplicatas por id_papel e data:", duplicatas_id_data)

Duplicatas por id_papel e data: 0


In [57]:
df_precos_setores = (
    df_precos_setores
    .sort_values(
        ["id_papel", "data", "volume_financeiro"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["id_papel", "data"], keep="first")
    .reset_index(drop=True)
)

print(
    "Duplicatas restantes:",
    df_precos_setores.duplicated(subset=["id_papel", "data"]).sum()
)

Duplicatas restantes: 0


### Salvamento da base de preços com setores

Nesta etapa, salvamos a base de preços já combinada com as informações cadastrais e setoriais.

In [58]:
arquivo_saida = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

df_precos_setores.to_parquet(arquivo_saida, index=False)

print("Base de preços com setores salva em:", arquivo_saida)

Base de preços com setores salva em: ..\dados_tratados\dados_economatica_B3_com_setores.parquet


In [59]:
df_teste = pd.read_parquet(arquivo_saida)

print("Base salva:", df_teste.shape)

display(df_teste.head())

Base salva: (1374399, 21)


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ABYA3,2010-01-04,ABYA3<XBSP>,4.6000,4.6100,4.5700,4.6200,4.6000,463.0000,"3,991,454.0000","868,600.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
1,ABYA3,2010-01-05,ABYA3<XBSP>,4.5800,4.6300,4.5700,4.6300,4.6000,319.0000,"3,280,211.0000","713,700.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
2,ABYA3,2010-01-06,ABYA3<XBSP>,4.8700,4.5700,4.5600,4.9200,4.8000,"1,715.0000","18,347,711.0000","3,826,000.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
3,ABYA3,2010-01-07,ABYA3<XBSP>,5.1700,4.7900,4.7400,5.1800,5.0300,"2,655.0000","22,244,229.0000","4,420,100.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
4,ABYA3,2010-01-08,ABYA3<XBSP>,5.4200,5.1800,5.1800,5.4500,5.3200,"2,229.0000","21,789,270.0000","4,093,100.0000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
